## Data Download

Download road Lines from OSM Overpass API

### Dependencies

In [1]:
import os

import geopandas as gpd
import requests as r
from shapely.geometry import LineString

from detect_wildlife_crossings.osm.queries import get_street_query

In [2]:
COUNTRIES_TO_PROCESS = ["CH"]  # List of country codes to process
GEODATA_OUTPUT_DIR = "../data/geodata/"  # Directory to save the output GPKGs, names will be generated as "<country>_wildlife_crossings_osm<bridges_string>.gpkg"
STREET_TYPE_DEPTH = (
    1  # how detailed the street network should be (see tag "highway" in OSM)
)

In [3]:
# these typically do not change
OVERPASS_API_URL = "https://overpass-api.de/api/interpreter"
CRS = "CRS:3035"  # target CRS for the output GPKGs
OVERPASS_TIMEOUT = 180  # timeout for overpass query in seconds

### Execution

In [4]:
for country in COUNTRIES_TO_PROCESS:
    print(f"Processing country for streets: {country}")

    # Download street data
    street_query = get_street_query(
        extent_country=country,
        street_type_depth=STREET_TYPE_DEPTH,
        timeout=OVERPASS_TIMEOUT,
    )
    response = r.get(OVERPASS_API_URL, params={"data": street_query})
    response.raise_for_status()
    data = response.json()

    features = []
    for el in data["elements"]:
        if el["type"] == "way" and "geometry" in el:
            coords = [(pt["lon"], pt["lat"]) for pt in el["geometry"]]
            # Only open ways → LineString
            if len(coords) >= 2 and coords[0] != coords[-1]:
                features.append(
                    {
                        "id": el["id"],
                        "geometry": LineString(coords),
                        **el.get("tags", {}),
                    }
                )

    # --- Build GeoDataFrame ---
    if not features:
        print("No LineString features found!")
        gdf = gpd.GeoDataFrame(columns=["id", "geometry"])
    else:
        gdf = gpd.GeoDataFrame(features, geometry="geometry", crs="EPSG:4326")
        print(f"Found {len(gdf)} LineString features.")

    # --- Prepare output folder ---
    output_dir = os.path.abspath(os.path.join(GEODATA_OUTPUT_DIR))
    os.makedirs(output_dir, exist_ok=True)

    # --- Reproject to EPSG:3035 (ETRS89 / LAEA Europe) ---
    if not gdf.empty:
        gdf = gdf.to_crs(epsg=CRS.split(":")[1])
        print(f"Reprojected to CRS:{CRS}.")

    # --- Save GPKG ---
    output_file = os.path.join(
        output_dir, f"{country}_highway_depth_{STREET_TYPE_DEPTH}.gpkg"
    )
    gdf.to_file(output_file, driver="GPKG")
    print(f"Data saved to {output_file}")

Processing country for streets: CH
Found 14716 LineString features.
Reprojected to CRS:CRS:3035.
Data saved to c:\code\cassda-zertifikatsarbeit\data\geodata\CH_highway_depth_1.gpkg
